# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [26]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [27]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [28]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/Projects_with_Domains.csv",
    metadata_columns=[
      "Project Title",
      "Project Domain",
      "Secondary Domain",
      "Description",
      "Judge Comments",
      "Score",
      "Project Name",
      "Judge Score"
    ]
)

synthetic_usecase_data = loader.load()

for doc in synthetic_usecase_data:
    doc.page_content = doc.metadata["Description"]

Let's look at an example document to see if everything worked as expected!

In [29]:
synthetic_usecase_data[0]

Document(metadata={'source': './data/Projects_with_Domains.csv', 'row': 0, 'Project Title': 'InsightAI 1', 'Project Domain': 'Security', 'Secondary Domain': 'Finance / FinTech', 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.', 'Judge Comments': 'Technically ambitious and well-executed.', 'Score': '85', 'Project Name': 'Project Aurora', 'Judge Score': '9.5'}, page_content='A low-latency inference system for multimodal agents in autonomous systems.')

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [30]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [31]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [32]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [33]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [34]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [35]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," which has multiple entries in the sample.'

In [36]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are usecases related to security. Specifically, one of the projects titled "Pathfinder 24" in the Healthcare / MedTech domain mentions improving sustainability, but also has a secondary domain of Security, indicating a focus on security aspects.'

In [37]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges\' comments on the fintech projects were generally positive. They described the work as solid, impressive, promising, and technically ambitious. Some specific comments include:\n\n- "Solid work with impressive real-world impact."\n- "Comprehensive and technically mature approach."\n- "Promising idea with robust experimental validation."\n- "Technically ambitious and well-executed."\n\nOverall, the judges appreciated the quality, validation, and potential impact of the fintech projects.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [38]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [39]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [40]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Finance / FinTech," as it is mentioned multiple times in the sample entries.'

In [41]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there was a use case related to security. The project "SecureNest 49" involved a document summarization and retrieval system for enterprise knowledge bases, which relates to security and compliance in enterprise environments.'

In [42]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive remarks about the fintech projects. Specifically, for the project "SynthMind," related to finance and fintech, the judge commented that it was "Conceptually strong but results need more benchmarking," and gave it a high score of 9.6. For the project "PulseAI 50," also in the fintech domain under secondary domain finance, the judge described it as "Technically ambitious and well-executed," with a score of 95. Overall, the judges viewed the fintech projects favorably, noting their strong concepts and technical quality.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer
BM25 is good in situations where we need classical search capabilities like scoring a document based on how many times a keyword occurred in a document. Its useful in situations where exact keyword match is required. For example question - Which projects used FastAPI?. In this case we want only projects that have fastAPI and not necessarily project that might be alternatives to fastapi like django or flask.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [43]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [44]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [45]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data is "Healthcare / MedTech," which appears multiple times. However, based solely on the sample, "Security" and "Creative / Design / Media" also appear. Since I only have a small sample, I cannot definitively determine the most common domain across all data. \n\nIf you need an accurate answer based on the entire dataset, I recommend analyzing the full dataset to count the occurrences of each domain.'

In [46]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no explicit use cases related to security. The use cases mentioned focus on federated learning for improving privacy in healthcare applications, which is related to privacy and data security, but not specifically about security use cases.'

In [47]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive comments about the fintech project "Pathfinder 27." They described it as having excellent code quality and making good use of open-source libraries.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [48]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [49]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [50]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain among the listed projects is "Healthcare / MedTech," appearing three times in the data.'

In [51]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, one project titled "Project Aurora" is described as a low-latency inference system for multimodal agents in autonomous systems, which falls under the security domain.'

In [52]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had a generally positive view of the fintech projects. For example, the project "Pathfinder" received an 81 score with comments highlighting "excellent code quality and use of open-source libraries." Overall, judges appreciated the quality, implementation, and potential impact of these fintech projects, as indicated by their favorable scores and remarks on the promising and well-executed nature of the ideas.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer
1) Multiple reformulations can help take into account different ways of describing the same concept. By using multiple reformulations that have synonyms and semantically related terms, the system can retrieve documents that do not contain the original query but may still be relevant to the user's intent.
2) Users may not always use the same terms and so having multiple formulations of the same query can help match documents that a single query cannot.
3) A single short query can be interpreted in multiple ways. Generating a response for each reformulation can help the system better respond to the search intent.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [53]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [54]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [55]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [56]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [57]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [58]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain is not explicitly stated as the most frequent in the sample. However, from the examples given, the project domains listed include Security, Creative / Design / Media, Productivity Assistants, and Healthcare / MedTech. Since there are only a few sample entries, I cannot determine definitively which is the most common overall. \n\nIf you have the full dataset, I recommend analyzing it for the most frequently occurring project domain.'

In [59]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases related to security mentioned. The projects listed focus on federated learning to improve privacy in healthcare, but there is no explicit mention of security-related use cases.'

In [60]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges\' comments regarding the fintech projects were positive. They described the projects as "technically ambitious," "well-executed," "comprehensive," "technically mature," and "a clever solution with measurable environmental benefit."'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [61]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [62]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [63]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data is "E‑commerce / Marketplaces," which appears multiple times among the listed projects.'

In [64]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, the project titled "SecureNest 28" involves a hardware-aware model quantization benchmark suite, which pertains to security and compliance aspects.'

In [65]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges\' comments on the fintech projects were generally positive regarding the quality and potential of the projects. For example:\n\n- The project "SecureNest 28" was described as "Conceptually strong but results need more benchmarking," with a high judge score of 9.0.\n- "Pathfinder 27" received praise for its "Excellent code quality and use of open-source libraries," and a judge score of 9.8.\n- "SecureNest 49" was called a "Comprehensive and technically mature approach," with a judge score of 9.2.\n- "MediMind 48" was acknowledged for "Great innovation but needs stronger evaluation metrics," with a judge score of 7.5.\n- "PulseAI 50" was described as "Technically ambitious and well-executed," with a judge score of 8.0.\n\nOverall, the judges recognized the strength, innovation, and technical quality of the fintech projects, though some noted areas for improvement such as benchmarking and evaluation metrics.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [66]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [67]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [68]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [69]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [70]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [71]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Legal / Compliance," which is mentioned twice among the sample projects. Other domains like "Developer Tools / DevEx" and "Healthcare / MedTech" are also referenced multiple times, but with less frequency in this sample.\n\nGiven this limited sample, I would say that "Legal / Compliance" seems to be a prominent domain among these projects. However, for a definitive answer, a broader dataset analysis would be necessary.'

In [72]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, one project titled "BioForge" is a medical imaging solution in the Security domain, and another project titled "SecureNest" is also in the Security domain, focusing on a low-latency inference system for multimodal agents in autonomous systems.'

In [73]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had the following comments about the fintech projects:\n\n- "Technically ambitious and well-executed." (for the project "TrendLens 19")\n- "Comprehensive and technically mature approach." (for "WealthifyAI 16")\n- "A forward-looking idea with solid supporting data." (for "AutoMate 5")'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer
if sentences are highly repetitive, we may end up with all similar senetences in one chunk leading to very large sized chunks which could lead to large context windows and more token usage (costly). We could add a max chuk size to limit the size of the chunks.

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [5]:
### YOUR CODE 
import os
from getpass import getpass
os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

In [ ]:
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")

# Set other required LangSmith environment variables
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "AIE8 Retrieval evaluation"  

In [6]:
# Loading data
from langchain_community.document_loaders.csv_loader import CSVLoader

csv_loader = CSVLoader(
    file_path=f"./data/Projects_with_Domains.csv",
    metadata_columns=[
      "Project Title",
      "Project Domain",
      "Secondary Domain",
      "Description",
      "Judge Comments",
      "Score",
      "Project Name",
      "Judge Score"
    ]
)

expanded_usecase_data = csv_loader.load()

In [7]:
expanded_usecase_data[0].metadata

{'source': './data/Projects_with_Domains.csv',
 'row': 0,
 'Project Title': 'InsightAI 1',
 'Project Domain': 'Security',
 'Secondary Domain': 'Finance / FinTech',
 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.',
 'Judge Comments': 'Technically ambitious and well-executed.',
 'Score': '85',
 'Project Name': 'Project Aurora',
 'Judge Score': '9.5'}

In [8]:
#use the metadat to expand the dataset as we are running into 100 token ragas error
for doc in expanded_usecase_data:
    # Extract all metadata
    title = doc.metadata.get("Project Title", "")
    domain = doc.metadata.get("Project Domain", "")
    secondary_domain = doc.metadata.get("Secondary Domain", "")
    description = doc.metadata.get("Description", "")
    judge_comments = doc.metadata.get("Judge Comments", "")
    score = doc.metadata.get("Score", "")
    project_name = doc.metadata.get("Project Name", "")
    judge_score = doc.metadata.get("Judge Score", "")
    
    # Create expanded content with more context
    expanded_content = f"""
    {doc.page_content}
    
    This project marks a notable advancement in the {domain} space. 
    The work, titled "{title}", centers on {description.lower()}.
    With an overall score of {score} and a judge score of {judge_score}, it demonstrates strong technical rigor and originality. 
    Judge feedback includes: {judge_comments}. 
    Its leadership in the {domain} domain, paired with a secondary emphasis on {secondary_domain}, underscores clear potential for real-world adoption and impact.
    Technically, the project’s approach and implementation reflect mature capabilities within {domain}. By combining engineering excellence with practical applicability, the effort stands out. The high scores and positive evaluations indicate meaningful momentum and strong potential for future development.

    """
    
    # Update the document
    doc.page_content = expanded_content

In [10]:
# Number of documents generated
len(expanded_usecase_data)

50

In [11]:
# Print document
expanded_usecase_data

[Document(metadata={'source': './data/Projects_with_Domains.csv', 'row': 0, 'Project Title': 'InsightAI 1', 'Project Domain': 'Security', 'Secondary Domain': 'Finance / FinTech', 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.', 'Judge Comments': 'Technically ambitious and well-executed.', 'Score': '85', 'Project Name': 'Project Aurora', 'Judge Score': '9.5'}, page_content='\n    \n\n    This project marks a notable advancement in the Security space. \n    The work, titled "InsightAI 1", centers on a low-latency inference system for multimodal agents in autonomous systems..\n    With an overall score of 85 and a judge score of 9.5, it demonstrates strong technical rigor and originality. \n    Judge feedback includes: Technically ambitious and well-executed.. \n    Its leadership in the Security domain, paired with a secondary emphasis on Finance / FinTech, underscores clear potential for real-world adoption and impact.\n    Technically, the p

In [13]:
# Check the length to ensure it meets RAGAS token requirements
len(expanded_usecase_data[0].page_content)

862

In [14]:
## Generate personas for ragas to use
from ragas.testset.persona import Persona

persona_novice= Persona(
    name="Novice builder",
    role_description="Don't know much about AI and is looking for information on how to get started on an AI project ",
)
persona_seasoned = Persona(
    name="Seasoned builder",
    role_description="Knows about AI and has built a few projects but is looking for more information on how to improve their skills",
)
persona_expert = Persona(
    name="Expert builder",
    role_description="Seasoned AI builder who has built many projects and is looking for more projects that they can build.",
)

personas = [persona_novice, persona_seasoned, persona_expert]
personas

/Users/priyankadogra/workspace/aimakerspace/code/AIE8/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/Users/priyankadogra/workspace/aimakerspace/code/AIE8/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/Users/priyankadogra/workspace/aimakerspace/code/AIE8/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


[Persona(name='Novice builder', role_description="Don't know much about AI and is looking for information on how to get started on an AI project "),
 Persona(name='Seasoned builder', role_description='Knows about AI and has built a few projects but is looking for more information on how to improve their skills'),
 Persona(name='Expert builder', role_description='Seasoned AI builder who has built many projects and is looking for more projects that they can build.')]

In [15]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

In [16]:
# We will use the SingleHopSpecificQuerySynthesizer to generate queries for our personas
from ragas.testset.synthesizers import SingleHopSpecificQuerySynthesizer
query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1),
    
]

In [17]:
# Create a knowledge graph
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

In [18]:
# Add document to knowledge graph
from ragas.testset.graph import Node, NodeType

### NOTICE: use the generated page_content and document_metadata from the expanded_usecase_data
for doc in expanded_usecase_data:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 50, relationships: 0)

In [20]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=expanded_usecase_data, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying SummaryExtractor:   0%|          | 0/50 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/50 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/150 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 50, relationships: 271)

In [24]:
# We will not generate the test dataset as it is already generated
from ragas.testset import TestsetGenerator
testset_generator = TestsetGenerator(knowledge_graph = kg, llm= generator_llm, embedding_model = generator_embeddings, persona_list= personas)
testset_based_onexpanded_usecase_data = testset_generator.generate(testset_size=10, query_distribution= query_distribution)

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

In [25]:
testset_based_onexpanded_usecase_data.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Wut makse InsightAI 1 stand out as a project i...,[\n \n\n This project marks a notable ad...,InsightAI 1 stands out in the Security space d...,single_hop_specifc_query_synthesizer
1,What is the focus of the ShopSmart 2 project i...,[\n \n\n This project marks a notable ad...,The ShopSmart 2 project centers on a simulatio...,single_hop_specifc_query_synthesizer
2,How does the WealthifyAI 3 project address Sec...,[\n \n\n This project marks a notable ad...,The WealthifyAI 3 project places a secondary e...,single_hop_specifc_query_synthesizer
3,what make E‑commerce / Marketplaces project go...,[\n \n\n This project marks a notable ad...,This E‑commerce / Marketplaces project is good...,single_hop_specifc_query_synthesizer
4,What is the role of bioinformatics in the Auto...,[\n \n\n This project marks a notable ad...,The AutoMate 5 project uses a bioinformatics p...,single_hop_specifc_query_synthesizer
5,what make this project good for DevEx? why it ...,[\n \n\n This project marks a notable ad...,This project is a notable advancement in the D...,single_hop_specifc_query_synthesizer
6,How does PlanPilot 7 address Legal / Complianc...,[\n \n\n This project marks a notable ad...,PlanPilot 7 places a secondary emphasis on Leg...,single_hop_specifc_query_synthesizer
7,i dont really get what InsightAI 8 is about an...,[\n \n\n This project marks a notable ad...,InsightAI 8 is a project focused on using a gr...,single_hop_specifc_query_synthesizer
8,What is ChatBridge 9 and how does it contribut...,[\n \n\n This project marks a notable ad...,ChatBridge 9 is a project that centers on a bi...,single_hop_specifc_query_synthesizer
9,how this Customer Support / Helpdesk project d...,[\n \n\n This project marks a notable ad...,"This Customer Support / Helpdesk project, call...",single_hop_specifc_query_synthesizer


## Redo the vector store based on the expanded data
We had to recreate some expanded data to ovecome the RAGAS token limit, so we now need to re-implement the vectore store and retrieval.

In [86]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore_ragas = Qdrant.from_documents(
    expanded_usecase_data,
    embeddings,
    location=":memory:",
    collection_name = 'Ragas_Expanded_Usecases'
)

## Add naive retrieval

In [87]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

# We will also use langsmith for the ragas evaluation

# set up ragas metrics to evaluate

In [89]:
from ragas.metrics import context_recall, faithfulness, context_precision

# Define the metrics you want to evaluate
metrics = [
    context_recall,
    faithfulness,
    context_precision
]

In [90]:
# Generate responses using naive retrieval for evaluation
import pandas as pd
from datasets import Dataset

# Convert test dataset to pandas
test_df = testset_based_onexpanded_usecase_data.to_pandas()
print(f"Test dataset has {len(test_df)} samples")

# Generate responses using naive retrieval chain
def generate_naive_responses(questions):
    responses = []
    contexts = []
    
    for i, question in enumerate(questions):
        print(f"Processing {i+1}/{len(questions)}: {question[:50]}...")
        try:
            result = naive_retrieval_chain.invoke({"question": question})
            response = result["response"].content
            context = "\n".join([doc.page_content for doc in result["context"]])
            
            responses.append(response)
            contexts.append(context)
        except Exception as e:
            print(f"Error: {e}")
            responses.append("Error generating response")
            contexts.append("No context available")
    
    return responses, contexts

print("\nGenerating responses with naive retrieval...")
responses, contexts = generate_naive_responses(test_df["user_input"].tolist())
print(f"Generated {len(responses)} responses!")

Test dataset has 10 samples

Generating responses with naive retrieval...
Processing 1/10: Wut makse InsightAI 1 stand out as a project in th...
Processing 2/10: What is the focus of the ShopSmart 2 project invol...
Processing 3/10: How does the WealthifyAI 3 project address Securit...
Processing 4/10: what make E‑commerce / Marketplaces project good h...
Processing 5/10: What is the role of bioinformatics in the AutoMate...
Processing 6/10: what make this project good for DevEx? why it matt...
Processing 7/10: How does PlanPilot 7 address Legal / Compliance co...
Processing 8/10: i dont really get what InsightAI 8 is about and wh...
Processing 9/10: What is ChatBridge 9 and how does it contribute to...
Processing 10/10: how this Customer Support / Helpdesk project do go...
Generated 10 responses!


In [91]:
# Format data for Ragas evaluation
naive_eval_data = {
    "question": test_df["user_input"].tolist(),
    "ground_truth": test_df["reference"].tolist(), 
    "answer": responses,
    "contexts": [[context] for context in contexts]
}

# Create Ragas dataset
naive_dataset = Dataset.from_dict(naive_eval_data)
print(f"Created Ragas dataset with {len(naive_dataset)} samples")
print("Dataset columns:", naive_dataset.column_names)

Created Ragas dataset with 10 samples
Dataset columns: ['question', 'ground_truth', 'answer', 'contexts']


In [94]:
# Run Ragas evaluation using the metrics defined above
from ragas import evaluate
import time

print("Starting Ragas evaluation for naive retriever...")
print(f"Using metrics: context_recall, faithfulness, context_precision")

start_time = time.time()

try:
    # Run evaluation using the metrics from the previous cell
    naive_results = evaluate(
        dataset=naive_dataset,
        metrics=metrics,  # Using the metrics list you defined above
    )
    
    end_time = time.time()
    evaluation_time = end_time - start_time
    
    print(f"\n✅ Evaluation completed in {evaluation_time:.2f} seconds")
    print("\nNaive Retrieval Results:")
    print("=" * 40)
    
    # Handle different types of results from Ragas
    print(f"Result type: {type(naive_results)}")
    
    if isinstance(naive_results, dict):
        # If it's a dictionary
        for metric_name, score in naive_results.items():
            print(f"{metric_name}: {score:.4f}")
    elif isinstance(naive_results, list):
        # If it's a list (newer Ragas versions)
        print(f"Results list: {naive_results}")
        # Try to extract scores if they're in a specific format
        for item in naive_results:
            print(f"Item: {item}")
    elif hasattr(naive_results, 'to_pandas'):
        # If it has a to_pandas method
        results_df = naive_results.to_pandas()
        print("Results DataFrame:")
        print(results_df)
    elif hasattr(naive_results, '__dict__'):
        # If it's an object with attributes
        for attr_name, attr_value in naive_results.__dict__.items():
            if isinstance(attr_value, (int, float)):
                print(f"{attr_name}: {attr_value:.4f}")
    else:
        # Fallback - just print the result
        print(f"Results: {naive_results}")
    
    # Store results for comparison later
    naive_evaluation_results = naive_results
    
    # Debug info
    print(f"\nDebugging info:")
    print(f"Type: {type(naive_results)}")
    if hasattr(naive_results, '__dict__'):
        print(f"Attributes: {list(naive_results.__dict__.keys())}")
    if hasattr(naive_results, '__len__'):
        print(f"Length: {len(naive_results)}")
    
except Exception as e:
    print(f"❌ Error during evaluation: {e}")
    print("Please check your setup and try again.")
    import traceback
    traceback.print_exc()

Starting Ragas evaluation for naive retriever...
Using metrics: context_recall, faithfulness, context_precision


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]


✅ Evaluation completed in 34.25 seconds

Naive Retrieval Results:
Result type: <class 'ragas.dataset_schema.EvaluationResult'>
Results DataFrame:
                                          user_input  \
0  Wut makse InsightAI 1 stand out as a project i...   
1  What is the focus of the ShopSmart 2 project i...   
2  How does the WealthifyAI 3 project address Sec...   
3  what make E‑commerce / Marketplaces project go...   
4  What is the role of bioinformatics in the Auto...   
5  what make this project good for DevEx? why it ...   
6  How does PlanPilot 7 address Legal / Complianc...   
7  i dont really get what InsightAI 8 is about an...   
8  What is ChatBridge 9 and how does it contribut...   
9  how this Customer Support / Helpdesk project d...   

                                  retrieved_contexts  \
0  [A low-latency inference system for multimodal...   
1  [A simulation environment for embodied AI agen...   
2  [A federated learning toolkit improving privac...   
3  [An AI-po

In [95]:
# Test LangSmith connection
from langsmith import Client

try:
    client = Client()
    print("✅ LangSmith client created successfully")
    
    # Test a simple trace
    from langchain_core.tracers import LangChainTracer
    tracer = LangChainTracer()
    print("✅ LangChain tracer created successfully")
    
except Exception as e:
    print(f"❌ LangSmith connection error: {e}")

✅ LangSmith client created successfully
✅ LangChain tracer created successfully
